# Phase 2 — assistant acts

Is the coach producing the same responses in phase 2 as it did in phase 1, and is it
responding to participant behaviour the same way?

This matters because the whole intervention logic runs through the assistant. If phase 2's
coach behaves differently, a transfer difference between the two studies could be the
coach rather than the participants.

**Provenance.** Assistant turns carry `annotation_assistant_fine`, written by
`../bad_user_sim/phase2_assistant_fine.py` using the same Othello-worded fine tutor-move
panel (`assistant_acts_othello.OthelloFineAssistantSuite`, three seats: gpt / llama /
sonnet) that produced the phase-1 labels. Re-run that script as the sample fills; it skips
turns that already carry the field. This notebook only reads.

**Scheme.** The revised scheme from `../bad_user_sim/elicitation_analysis.ipynb`: the three
affective codes collapse to one `Feedback`, and `Comprehension Gauging Question` and
`Paraphrase` are dropped. Collapsing is applied to each seat *before* the majority vote,
not to `final` after it, so two seats that disagree on the flavour of feedback still agree
that feedback occurred.

**Reliability.** Six codes clear kappa .6 and can carry a claim: Move Verdict, Prompt,
Board Report, General Principle, Worked Line, Local Justification. `Hint` (.38) and
`Feedback` (.47) do not. They stay in the tables, marked, so a reader can see them without
being invited to conclude from them.

**What each section answers**

- **A** is the coach itself the same — model and system prompt
- **B** does it produce the same acts at the same rates
- **C** does the same participant behaviour elicit the same acts (the elicitation structure)
- **D** do the two studies agree, cell by cell
- **E** does the arm change what the assistant gives

## 0. Setup

In [1]:
import hashlib
import json
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

BASE = Path.cwd()
CITRUS = BASE.parent
P1_REC = CITRUS / "game_user_study_phase_othello" / "recordings-download"
P2_REC = BASE / "recordings-download"

AI = "poc20260727"                      # the assisted round, same puzzle in both studies
ANN = f"annotated_conversation_{AI}.jsonl"
FA, FP = "annotation_assistant_fine", "annotation_user"

PACTS = ["Knowledge Deficit Question", "Solution Request", "Think Aloud",
         "Common Ground Question", "Conversational Acknowledgment", "Metacomment"]
SHORT = {"Knowledge Deficit Question": "KDQ", "Solution Request": "SolReq",
         "Think Aloud": "ThinkAloud", "Common Ground Question": "CGQ",
         "Conversational Acknowledgment": "ConvAck", "Metacomment": "Metacomment"}

# The three participant acts the intervention targets or is measured against.
FOCUS = ["Think Aloud", "Solution Request", "Knowledge Deficit Question"]

ALL_A = ["Board Report", "Move Verdict", "General Principle", "Local Justification",
         "Worked Line", "Prompt", "Hint", "Feedback"]
# Clears kappa .6 on the revised scheme. Hint (.38) and Feedback (.47) do not.
RELIABLE = {"Move Verdict", "Prompt", "Board Report", "General Principle",
            "Worked Line", "Local Justification"}

SEATS = ["gpt", "llama", "sonnet"]
COLLAPSE = {"Positive Feedback": "Feedback", "Negative Feedback": "Feedback",
            "Neutral Feedback": "Feedback"}
DROP = {"Comprehension Gauging Question", "Paraphrase"}

In [2]:
def remap(labels):
    return {COLLAPSE.get(c, c) for c in labels if c not in DROP}


def revote(rec, min_votes=2):
    """Re-derive the panel label under the revised scheme.

    Collapsing is applied per seat and then re-voted, so a turn where one seat says
    Positive and another says Neutral still lands on Feedback. Falls back to the stored
    `final` when a seat is missing.
    """
    per = {k: v for k, v in (rec.get("per_model") or {}).items() if v is not None}
    if len(per) < len(SEATS):
        return remap(rec.get("final") or []), len(per)
    cnt = Counter()
    for s in SEATS:
        cnt.update(remap(per[s]))
    return {c for c, n in cnt.items() if n >= min_votes}, len(per)


def seat_agreement(rec):
    """Mean pairwise Jaccard across seats -- a reliability check on the revised scheme."""
    per = [remap(v) for v in (rec.get("per_model") or {}).values() if v is not None]
    if len(per) < 2:
        return np.nan
    out = []
    for i in range(len(per)):
        for j in range(i + 1, len(per)):
            u = per[i] | per[j]
            out.append(1.0 if not u else len(per[i] & per[j]) / len(u))
    return float(np.mean(out))


def load(root, with_arm=False):
    """One row per ASSISTANT turn, carrying both annotation layers. Cluster = pid."""
    rows = []
    for d in sorted(Path(root).iterdir()):
        f = d / ANN
        if not d.is_dir() or not f.exists():
            continue
        arm = None
        if with_arm:
            arm = ("scaffolded" if (d / "intervention.json").exists()
                   else "vanilla" if (d / "filler.json").exists() else None)
        for line in open(f):
            if not line.strip():
                continue
            t = json.loads(line)
            if FA not in t or not (t.get("assistant") or "").strip():
                continue
            asst, nseat = revote(t[FA])
            p = set((t.get(FP) or {}).get("final") or [])
            r = dict(pid=d.name, arm=arm, asst=asst, nseat=nseat,
                     agree=seat_agreement(t[FA]), awords=len(t["assistant"].split()))
            for a in PACTS:
                r[SHORT[a]] = int(a in p)
            rows.append(r)
    return pd.DataFrame(rows)


P1 = load(P1_REC)
P2 = load(P2_REC, with_arm=True)
print(f"phase 1: {len(P1):>4} assistant turns, {P1.pid.nunique()} participants")
print(f"phase 2: {len(P2):>4} assistant turns, {P2.pid.nunique()} participants "
      f"(arm known for {int(P2.arm.notna().sum())})")
print(f"\nthree-seat turns: phase 1 {100*(P1.nseat==3).mean():.0f}%, "
      f"phase 2 {100*(P2.nseat==3).mean():.0f}%")
print(f"panel agreement (mean pairwise Jaccard): phase 1 {P1.agree.mean():.3f}, "
      f"phase 2 {P2.agree.mean():.3f}")
print("\n  Agreement at or above phase 1 means a prevalence difference below is not an\n"
      "  annotation artefact.")

phase 1:  667 assistant turns, 120 participants
phase 2:  388 assistant turns, 80 participants (arm known for 388)

three-seat turns: phase 1 100%, phase 2 100%
panel agreement (mean pairwise Jaccard): phase 1 0.750, phase 2 0.771

  Agreement at or above phase 1 means a prevalence difference below is not an
  annotation artefact.


## B. Does the coach produce the same acts?

Prevalence is the share of assistant turns carrying each act. Turns are the unit, so a
participant who sent more messages contributes more; that is the right unit for "what does
a reply look like", and the wrong one for "what did a participant receive". Section E uses
the same unit, so the arm comparison is internally consistent.

In [3]:
rows = []
for a in ALL_A:
    x = P1.asst.apply(lambda s, t=a: int(t in s))
    y = P2.asst.apply(lambda s, t=a: int(t in s))
    tab = np.array([[x.sum(), len(x) - x.sum()], [y.sum(), len(y) - y.sum()]])
    p = stats.chi2_contingency(tab)[1]
    rows.append((a, 100 * x.mean(), 100 * y.mean(), p))

t = pd.DataFrame(rows, columns=["act", "p1", "p2", "p"]).sort_values("p")
k = len(t)
t["q"] = np.minimum.accumulate((t.p.values * k / np.arange(1, k + 1))[::-1])[::-1]
t = t.set_index("act").loc[ALL_A].reset_index()

print(f"{'assistant act':<22}{'phase 1 %':>11}{'phase 2 %':>11}{'diff':>8}{'p':>9}"
      f"{'q(BH)':>8}{'kappa>=.6':>11}")
for _, r in t.iterrows():
    star = "  *" if r.p < .05 else (" ." if r.p < .10 else "")
    print(f"{r['act']:<22}{r.p1:>11.1f}{r.p2:>11.1f}{r.p2-r.p1:>+8.1f}{r.p:>9.4f}"
          f"{r.q:>8.3f}{('yes' if r['act'] in RELIABLE else 'NO'):>11}{star}")

print(f"\n{int((t.p<.05).sum())} of {k} acts differ at raw p<.05; smallest BH q {t.q.min():.3f}")
print(f"\nacts per assistant turn:  phase 1 {P1.asst.apply(len).mean():.2f}   "
      f"phase 2 {P2.asst.apply(len).mean():.2f}   "
      f"MW p {stats.mannwhitneyu(P1.asst.apply(len), P2.asst.apply(len)).pvalue:.3f}")
print(f"assistant reply length:   phase 1 {P1.awords.mean():.0f} words   "
      f"phase 2 {P2.awords.mean():.0f} words   "
      f"MW p {stats.mannwhitneyu(P1.awords, P2.awords).pvalue:.3f}")
print("\n  Board Report is the act that predicted transfer in phase 1, so its direction\n"
      "  here is the one worth reading first.")

assistant act           phase 1 %  phase 2 %    diff        p   q(BH)  kappa>=.6
Board Report                 39.7       47.2    +7.4   0.0219   0.044        yes  *
Move Verdict                 42.4       45.6    +3.2   0.3456   0.346        yes
General Principle            59.2       54.1    -5.1   0.1213   0.162        yes
Local Justification          75.3       66.5    -8.8   0.0028   0.011        yes  *
Worked Line                  12.1        7.7    -4.4   0.0317   0.051        yes  *
Prompt                       30.0       22.7    -7.3   0.0125   0.033        yes  *
Hint                         25.8       22.4    -3.4   0.2501   0.286         NO
Feedback                     30.6       20.6   -10.0   0.0006   0.005         NO  *

5 of 8 acts differ at raw p<.05; smallest BH q 0.005

acts per assistant turn:  phase 1 3.15   phase 2 2.87   MW p 0.001
assistant reply length:   phase 1 61 words   phase 2 53 words   MW p 0.000

  Board Report is the act that predicted transfer in phase

## C. Elicitation — what participant behaviour summons what

One logit per assistant act, on all six participant acts simultaneously, standard errors
clustered by participant. The adjustment matters: participant acts co-occur within a turn,
so an unadjusted lift for one act partly reflects the others.

These are **observational**. The participant's act is not randomised, so an elicitation
association is not a causal effect. Forcing an act does not reproduce it -- in the
simulator's `gate-require_kdq` arm the act was classifier-enforced and the link it was
supposed to carry collapsed.

In [4]:
def or_table(d, label):
    """Odds ratio per assistant act for each focus participant act, adjusted for all six."""
    print("=" * 92)
    print(f"{label} -- odds ratio for each assistant act, adjusted for all participant acts")
    print("=" * 92)
    print(f"{'assistant act':<22}" + "".join(f"{SHORT[a]:>22}" for a in FOCUS))
    res = {}
    for aa in ALL_A:
        y = d.asst.apply(lambda s, t=aa: int(t in s))
        if y.nunique() < 2:
            print(f"{aa:<22}  (no variation)")
            continue
        X = sm.add_constant(d[[SHORT[p] for p in PACTS]])
        try:
            m = sm.Logit(y, X).fit(disp=0, cov_type="cluster",
                                   cov_kwds={"groups": d.pid})
        except Exception as e:
            print(f"{aa:<22}  (did not converge: {type(e).__name__})")
            continue
        line = f"{aa:<22}"
        for pa in FOCUS:
            kk = SHORT[pa]
            o, pv = np.exp(m.params[kk]), m.pvalues[kk]
            star = "***" if pv < .001 else "**" if pv < .01 else "*" if pv < .05 else ""
            line += f"{o:>17.2f} {star:<4}"
            res[(aa, kk)] = (o, pv)
        print(line)
    print()
    return res


r1 = or_table(P1, "PHASE 1 OTHELLO")
r2 = or_table(P2, "PHASE 2")

PHASE 1 OTHELLO -- odds ratio for each assistant act, adjusted for all participant acts
assistant act                     ThinkAloud                SolReq                   KDQ
Board Report                       0.98                  0.17 ***              1.55     
Move Verdict                       0.72                 33.51 ***              0.11 *** 
General Principle                  1.00                  0.56 *                3.86 *** 
Local Justification                1.71                  7.71 ***              0.64     
Worked Line                        2.26 **               0.82                  0.63     
Prompt                             2.13 ***              0.15 ***              0.43 **  
Hint                               1.68 *                0.16 ***              0.42 *   
Feedback                           2.54 ***              0.28 ***              0.66     

PHASE 2 -- odds ratio for each assistant act, adjusted for all participant acts
assistant act                 

## D. Do the two studies agree?

Direction agreement is restricted to the reliable acts. A disagreement on a cell that is
null in both studies is not a reversal, so read the odds ratios alongside the verdict
rather than counting the verdict column alone.

In [5]:
print(f"{'assistant act':<22}{'participant act':<14}{'phase 1 OR':>12}{'phase 2 OR':>12}"
      f"{'same side of 1':>16}")
agree = []
for aa in ALL_A:
    if aa not in RELIABLE:
        continue
    for pa in FOCUS:
        kk = SHORT[pa]
        if (aa, kk) not in r1 or (aa, kk) not in r2:
            continue
        o1, o2 = r1[(aa, kk)][0], r2[(aa, kk)][0]
        same = (o1 > 1) == (o2 > 1)
        agree.append(same)
        print(f"{aa:<22}{kk:<14}{o1:>12.2f}{o2:>12.2f}"
              f"{('yes' if same else 'NO'):>16}")

print(f"\ndirection agreement on reliable acts: {sum(agree)}/{len(agree)}")

shared = [k for k in r1 if k in r2]
lo1 = np.log([r1[k][0] for k in shared])
lo2 = np.log([r2[k][0] for k in shared])
ok = np.isfinite(lo1) & np.isfinite(lo2)
print(f"correlation of log odds ratios across all {int(ok.sum())} cells: "
      f"r {np.corrcoef(lo1[ok], lo2[ok])[0, 1]:+.3f}")
print("\n  The pattern to check: Solution Request should summon Move Verdict and suppress\n"
      "  Board Report; Think Aloud should elicit Prompt and Hint while staying FLAT on\n"
      "  Board Report. That flatness is why Think Aloud's transfer effect cannot run\n"
      "  through the assistant.")

assistant act         participant act  phase 1 OR  phase 2 OR  same side of 1
Board Report          ThinkAloud            0.98        0.79             yes
Board Report          SolReq                0.17        0.11             yes
Board Report          KDQ                   1.55        1.51             yes
Move Verdict          ThinkAloud            0.72        0.56             yes
Move Verdict          SolReq               33.51        9.88             yes
Move Verdict          KDQ                   0.11        0.25             yes
General Principle     ThinkAloud            1.00        1.37             yes
General Principle     SolReq                0.56        0.99             yes
General Principle     KDQ                   3.86        2.17             yes
Local Justification   ThinkAloud            1.71        2.53             yes
Local Justification   SolReq                7.71        6.32             yes
Local Justification   KDQ                   0.64        0.44             ye

## E. Does the arm change what the assistant gives?

Both arms play the same round with the same coach, so a difference here would have to come
from the arms asking for different things. Phase-1 work found assistant acts are downstream
of participant help-seeking, which predicts close to nothing in this table.

In [6]:
A = P2[P2.arm.notna()]
print(f"turns: scaffolded {int((A.arm=='scaffolded').sum())}, "
      f"vanilla {int((A.arm=='vanilla').sum())}\n")
print(f"{'assistant act':<22}{'scaffolded %':>14}{'vanilla %':>11}{'diff':>8}{'p':>9}"
      f"{'kappa>=.6':>11}")
for a in ALL_A:
    s = A.loc[A.arm == "scaffolded", "asst"].apply(lambda x, t=a: int(t in x))
    v = A.loc[A.arm == "vanilla", "asst"].apply(lambda x, t=a: int(t in x))
    tab = np.array([[s.sum(), len(s) - s.sum()], [v.sum(), len(v) - v.sum()]])
    p = stats.chi2_contingency(tab)[1]
    star = "  *" if p < .05 else (" ." if p < .10 else "")
    print(f"{a:<22}{100*s.mean():>14.1f}{100*v.mean():>11.1f}"
          f"{100*(s.mean()-v.mean()):>+8.1f}{p:>9.3f}"
          f"{('yes' if a in RELIABLE else 'NO'):>11}{star}")
print("\n  Turns are the unit, so an arm that sent more messages contributes more turns.\n"
      "  Check the turn counts above before reading a small difference.")

turns: scaffolded 214, vanilla 174

assistant act           scaffolded %  vanilla %    diff        p  kappa>=.6
Board Report                    48.1       46.0    +2.2    0.749        yes
Move Verdict                    43.0       48.9    -5.9    0.294        yes
General Principle               54.2       54.0    +0.2    1.000        yes
Local Justification             67.3       65.5    +1.8    0.795        yes
Worked Line                      6.5        9.2    -2.7    0.434        yes
Prompt                          23.4       21.8    +1.5    0.814        yes
Hint                            25.7       18.4    +7.3    0.111         NO
Feedback                        24.8       15.5    +9.2    0.035         NO  *

  Turns are the unit, so an arm that sent more messages contributes more turns.
  Check the turn counts above before reading a small difference.


## Reading order

1. **A first.** If the model or the prompt differs, B is not interpretable as a
   participant-side difference.
2. **B** for whether the coach's output profile moved between studies. Board Report is the
   act that carried transfer in phase 1, so read its direction before the others.
3. **C and D** for whether the elicitation structure replicates. This is the part that
   supports or undercuts the intervention's premise, and it is measured on turns rather
   than participants, so it is far better powered than anything in `phase2_analysis.ipynb`.
4. **E last**, and expect nothing. A difference here would mean the arms are receiving
   different teaching, which would confound the primary endpoint.

Ignore `Hint` and `Feedback` when forming a conclusion. They are below the kappa threshold
and are printed only so their absence is visible rather than silent.